# 2ª Prova — Fundamentos em Ciência de Dados

**Nomes da dupla:** preencher aqui  
**Números USP:** preencher aqui  

Este notebook responde à prova usando uma análise exploratória de dados, com tabelas de frequência, medidas descritivas, histogramas, boxplots, gráficos de barras, gráficos de setores, gráficos de dispersão e gráficos de violino, conforme o escopo das aulas, com foco na AULA9.

**Observação metodológica:** esta análise é exploratória. Portanto, as conclusões indicam evidências visuais e descritivas de associação, mas não realizam testes de hipótese, teste t, qui-quadrado ou modelos inferenciais.

In [ ]:
# Bibliotecas usadas nas aulas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set(style='whitegrid')

## 1. Leitura do banco de dados

O conjunto de dados contém informações de funcionários e será usado para investigar, de forma exploratória, quais características parecem estar associadas a níveis mais altos ou mais baixos de produtividade.

In [ ]:
# No Colab, coloque o arquivo DadoseDecisoes.csv no ambiente ou ajuste o caminho abaixo.
df = pd.read_csv('DadoseDecisoes.csv')

# Visualização inicial
df.head()

In [ ]:
# Dimensão da base e tipos lidos pelo pandas
df.shape, df.dtypes

In [ ]:
# Verificação de valores ausentes
df.isna().sum()

A base possui 230 observações e 11 variáveis originais. Não foram observados valores ausentes nas variáveis do arquivo. A variável `ID` é apenas um identificador dos funcionários e, por isso, não será interpretada como variável explicativa da produtividade.

## 2. Questão (a): criação da variável `Produtividade_bin`

A classificação solicitada pela prova é:

- **Alta produtividade:** funcionários com produtividade acima da mediana;
- **Baixa produtividade:** funcionários com produtividade abaixo ou igual à mediana.

Assim, a mediana é usada como ponto de corte. Como o grupo “Baixa” inclui os valores iguais à mediana, os grupos não precisam ter exatamente o mesmo tamanho.

In [ ]:
mediana_produtividade = df['Produtividade'].median()
mediana_produtividade

In [ ]:
df['Produtividade_bin'] = np.where(
    df['Produtividade'] > mediana_produtividade,
    'Alta',
    'Baixa'
)

pd.crosstab(index=df['Produtividade_bin'], columns='Frequência')

In [ ]:
pd.crosstab(index=df['Produtividade_bin'], columns='Frequência relativa', normalize='columns')

A mediana observada de `Produtividade` é **87,04**. Com essa regra, foram classificados **114 funcionários como Alta produtividade** e **116 como Baixa produtividade**.

## 3. Questão (b): tipo de cada variável

A tabela abaixo classifica cada variável segundo a natureza estatística: quantitativa discreta, quantitativa contínua, qualitativa nominal ou qualitativa ordinal.

In [ ]:
tipos_variaveis = pd.DataFrame({
    'Variável': [
        'ID', 'Idade', 'Gênero', 'Departamento', 'Salário',
        'Horas_Trabalhadas', 'Produtividade', 'Satisfação',
        'Tempo_Empresa', 'Cursos_Realizados', 'Home_Office',
        'Produtividade_bin'
    ],
    'Tipo estatístico': [
        'Identificador',
        'Quantitativa discreta',
        'Qualitativa nominal',
        'Qualitativa nominal',
        'Quantitativa contínua',
        'Quantitativa discreta',
        'Quantitativa contínua',
        'Quantitativa contínua',
        'Quantitativa discreta',
        'Quantitativa discreta',
        'Qualitativa nominal',
        'Qualitativa ordinal'
    ],
    'Justificativa': [
        'Código de identificação; não representa uma medida de interesse.',
        'Contagem em anos completos.',
        'Categorias sem ordenação natural.',
        'Categorias sem ordenação natural.',
        'Medida monetária em escala contínua.',
        'Contagem de horas trabalhadas.',
        'Índice numérico medido em escala contínua.',
        'Índice numérico medido em escala contínua.',
        'Tempo contado em anos.',
        'Contagem de cursos realizados.',
        'Categorias Sim/Não, sem ordenação de intensidade.',
        'Categorias Baixa/Alta, com ordenação natural.'
    ]
})

tipos_variaveis

## 4. Questão (c): análise das variáveis qualitativas

Para as variáveis qualitativas, serão usadas tabelas de frequência absoluta, frequência relativa e gráficos de barras/setores. As variáveis qualitativas originais são `Gênero`, `Departamento` e `Home_Office`. A variável criada `Produtividade_bin` também é qualitativa ordinal.

In [ ]:
qualitativas = ['Gênero', 'Departamento', 'Home_Office', 'Produtividade_bin']

for var in qualitativas:
    print('\nVariável:', var)
    display(pd.crosstab(index=df[var], columns='Frequência'))
    display((pd.crosstab(index=df[var], columns='Frequência relativa', normalize='columns') * 100).round(2))

In [ ]:
for var in qualitativas:
    tab = pd.crosstab(index=df[var], columns='Frequência')
    ax = tab.plot.bar(figsize=(8, 4), legend=False, edgecolor='black')
    plt.title(f'Gráfico de barras — {var}')
    plt.xlabel(var)
    plt.ylabel('Frequência')
    plt.xticks(rotation=45, ha='right')
    plt.show()

In [ ]:
# Gráficos de setores para variáveis qualitativas
for var in qualitativas:
    tab = pd.crosstab(index=df[var], columns='Frequência')
    tab.plot.pie(
        y='Frequência',
        autopct='%1.1f%%',
        figsize=(6, 6),
        legend=False,
        startangle=90
    )
    plt.title(f'Gráfico de setores — {var}')
    plt.ylabel('')
    plt.show()

Pelas frequências gerais, observa-se que a base contém 102 funcionários do gênero feminino e 128 do gênero masculino. Em `Departamento`, a maior frequência é de TI, seguida por Vendas, Marketing e RH. A variável `Home_Office` está perfeitamente balanceada na amostra, com 115 funcionários em home-office e 115 sem home-office.

## 5. Questão (d): análise das variáveis quantitativas

Para variáveis quantitativas, serão usadas medidas descritivas e visualizações como histogramas, boxplots, gráficos de violino e dispersão.

In [ ]:
quantitativas = [
    'Idade', 'Salário', 'Horas_Trabalhadas', 'Produtividade',
    'Satisfação', 'Tempo_Empresa', 'Cursos_Realizados'
]

round(df[quantitativas].describe(), 2)

In [ ]:
# Histogramas
for var in quantitativas:
    sns.displot(df[var], bins=15, height=4, aspect=1.6)
    plt.title(f'Histograma — {var}')
    plt.xlabel(var)
    plt.ylabel('Frequência')
    plt.show()

In [ ]:
# Boxplots individuais
for var in quantitativas:
    plt.figure(figsize=(8, 3))
    sns.boxplot(x=df[var])
    plt.title(f'Boxplot — {var}')
    plt.xlabel(var)
    plt.show()

In [ ]:
# Gráficos de dispersão entre Produtividade e variáveis quantitativas
for var in ['Horas_Trabalhadas', 'Satisfação', 'Tempo_Empresa', 'Cursos_Realizados', 'Idade', 'Salário']:
    plt.figure(figsize=(7, 4))
    sns.scatterplot(x=df[var], y=df['Produtividade'], hue=df['Produtividade_bin'])
    plt.title(f'Produtividade versus {var}')
    plt.xlabel(var)
    plt.ylabel('Produtividade')
    plt.legend(title='Produtividade_bin')
    plt.show()

A variável `Produtividade` tem média aproximadamente igual a 87,85 e mediana igual a 87,04. `Horas_Trabalhadas` varia de 30 a 48 horas, com média próxima de 38,85 horas. `Salário` apresenta maior dispersão e assimetria do que as demais variáveis, pois há valores muito superiores à mediana.

## 6. Questão (e): tabelas resumo e gráficos de associação com `Produtividade_bin`

Nesta etapa, a variável criada `Produtividade_bin` é comparada com as demais variáveis do conjunto de dados. Para variáveis quantitativas, serão comparadas medidas descritivas por grupo. Para variáveis qualitativas, serão construídas tabelas de dupla entrada e gráficos de barras empilhadas.

In [ ]:
# Tabela resumo das quantitativas por Produtividade_bin
resumo_quantitativas_por_grupo = df.groupby('Produtividade_bin')[quantitativas].agg(
    ['count', 'mean', 'median', 'std', 'min', 'max']
).round(2)

resumo_quantitativas_por_grupo

In [ ]:
# Versão mais compacta para as principais variáveis de interesse da prova
principais_quantitativas = ['Horas_Trabalhadas', 'Satisfação', 'Tempo_Empresa', 'Cursos_Realizados']

resumo_principal = df.groupby('Produtividade_bin')[principais_quantitativas].agg(
    ['mean', 'median', 'std']
).round(2)

resumo_principal

In [ ]:
# Boxplots por grupo de produtividade
for var in ['Horas_Trabalhadas', 'Satisfação', 'Tempo_Empresa', 'Cursos_Realizados', 'Idade', 'Salário']:
    plt.figure(figsize=(7, 4))
    sns.boxplot(x='Produtividade_bin', y=var, data=df, order=['Baixa', 'Alta'])
    plt.title(f'{var} por grupo de produtividade')
    plt.xlabel('Produtividade_bin')
    plt.ylabel(var)
    plt.show()

In [ ]:
# Gráficos de violino por grupo de produtividade
for var in ['Horas_Trabalhadas', 'Satisfação', 'Tempo_Empresa', 'Cursos_Realizados']:
    plt.figure(figsize=(7, 4))
    sns.violinplot(x='Produtividade_bin', y=var, data=df, order=['Baixa', 'Alta'])
    plt.title(f'Gráfico de violino — {var} por Produtividade_bin')
    plt.xlabel('Produtividade_bin')
    plt.ylabel(var)
    plt.show()

In [ ]:
# Tabelas de dupla entrada para variáveis qualitativas
for var in ['Gênero', 'Departamento', 'Home_Office']:
    print('\nVariável:', var)
    print('Frequências absolutas')
    display(pd.crosstab(index=df[var], columns=df['Produtividade_bin'], margins=True))
    print('Percentuais por linha')
    display((pd.crosstab(index=df[var], columns=df['Produtividade_bin'], normalize='index') * 100).round(2))

In [ ]:
# Gráficos de barras empilhadas: distribuição de produtividade dentro de cada categoria
for var in ['Gênero', 'Departamento', 'Home_Office']:
    tabela = pd.crosstab(index=df[var], columns=df['Produtividade_bin'], normalize='index') * 100
    tabela = tabela[['Baixa', 'Alta']]
    tabela.plot.bar(stacked=True, figsize=(8, 4), edgecolor='black')
    plt.title(f'Percentual de produtividade por {var}')
    plt.xlabel(var)
    plt.ylabel('Percentual dentro da categoria')
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Produtividade_bin')
    plt.show()

As tabelas e gráficos indicam uma separação muito forte entre os grupos de produtividade para `Horas_Trabalhadas` e `Home_Office`. Para `Horas_Trabalhadas`, o grupo de alta produtividade possui média de 43,79 horas, enquanto o grupo de baixa produtividade possui média de 34,00 horas. Para `Home_Office`, praticamente todos os funcionários em home-office aparecem no grupo de alta produtividade, enquanto os funcionários que não trabalham em home-office aparecem no grupo de baixa produtividade.

## 7. Questão (f): investigação dos pontos solicitados

A seguir, cada ponto solicitado no enunciado é analisado de forma exploratória.

### 7.1 Funcionários com alta produtividade trabalham mais horas?

In [ ]:
df.groupby('Produtividade_bin')['Horas_Trabalhadas'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(2)

In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(x='Produtividade_bin', y='Horas_Trabalhadas', data=df, order=['Baixa', 'Alta'])
plt.title('Horas trabalhadas por grupo de produtividade')
plt.xlabel('Produtividade_bin')
plt.ylabel('Horas_Trabalhadas')
plt.show()

Sim. A evidência exploratória é forte: o grupo de alta produtividade tem média de **43,79 horas**, enquanto o grupo de baixa produtividade tem média de **34,00 horas**. Além disso, os boxplots mostram pouca sobreposição entre os grupos, indicando que `Horas_Trabalhadas` parece estar fortemente associada à classificação de produtividade.

### 7.2 Funcionários com alta produtividade possuem maior satisfação?

In [ ]:
df.groupby('Produtividade_bin')['Satisfação'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(2)

In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(x='Produtividade_bin', y='Satisfação', data=df, order=['Baixa', 'Alta'])
plt.title('Satisfação por grupo de produtividade')
plt.xlabel('Produtividade_bin')
plt.ylabel('Satisfação')
plt.show()

Não há evidência exploratória clara de que os funcionários de alta produtividade tenham maior satisfação. A média de satisfação é **85,50** no grupo de alta produtividade e **86,79** no grupo de baixa produtividade. As medianas são muito próximas: **87,82** para alta produtividade e **87,57** para baixa produtividade. Portanto, a satisfação não parece diferenciar fortemente os grupos.

### 7.3 Funcionários com alta produtividade permanecem mais tempo na empresa?

In [ ]:
df.groupby('Produtividade_bin')['Tempo_Empresa'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(2)

In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(x='Produtividade_bin', y='Tempo_Empresa', data=df, order=['Baixa', 'Alta'])
plt.title('Tempo de empresa por grupo de produtividade')
plt.xlabel('Produtividade_bin')
plt.ylabel('Tempo_Empresa')
plt.show()

Não há evidência forte nesse sentido. A média de tempo de empresa é **6,83 anos** para alta produtividade e **6,72 anos** para baixa produtividade, e as medianas são iguais a **6 anos** nos dois grupos. Assim, `Tempo_Empresa` parece ter associação fraca ou praticamente inexistente com a produtividade binária nesta amostra.

### 7.4 Funcionários com alta produtividade realizam mais cursos?

In [ ]:
df.groupby('Produtividade_bin')['Cursos_Realizados'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(2)

In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(x='Produtividade_bin', y='Cursos_Realizados', data=df, order=['Baixa', 'Alta'])
plt.title('Cursos realizados por grupo de produtividade')
plt.xlabel('Produtividade_bin')
plt.ylabel('Cursos_Realizados')
plt.show()

Há uma diferença pequena a favor do grupo de alta produtividade: média de **5,51 cursos** contra **4,91 cursos** no grupo de baixa produtividade. A mediana também é um pouco maior no grupo de alta produtividade: **5 cursos** contra **4 cursos**. Entretanto, a sobreposição entre os grupos é grande, então essa associação parece fraca quando comparada a `Horas_Trabalhadas` e `Home_Office`.

### 7.5 Funcionários de alta produtividade atuam em determinados departamentos?

In [ ]:
# Frequências absolutas por departamento e produtividade
pd.crosstab(index=df['Departamento'], columns=df['Produtividade_bin'], margins=True)

In [ ]:
# Percentuais por departamento
(pd.crosstab(index=df['Departamento'], columns=df['Produtividade_bin'], normalize='index') * 100).round(2)

In [ ]:
tabela_dep = pd.crosstab(index=df['Departamento'], columns=df['Produtividade_bin'], normalize='index') * 100
tabela_dep = tabela_dep[['Baixa', 'Alta']]
tabela_dep.plot.bar(stacked=True, figsize=(8, 4), edgecolor='black')
plt.title('Produtividade por departamento')
plt.xlabel('Departamento')
plt.ylabel('Percentual dentro do departamento')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Produtividade_bin')
plt.show()

A associação com departamento parece moderada a fraca. O departamento de RH apresenta maior proporção de baixa produtividade: **60,00%** dos funcionários de RH estão no grupo de baixa produtividade. Marketing, TI e Vendas apresentam proporções próximas de 50% em cada grupo, com ligeira maioria de alta produtividade. Portanto, o departamento pode sugerir alguma diferença, principalmente para RH, mas não há separação tão forte quanto em `Horas_Trabalhadas` ou `Home_Office`.

### 7.6 Funcionários com alta produtividade trabalham em regime de home-office?

In [ ]:
# Frequências absolutas por Home_Office e produtividade
pd.crosstab(index=df['Home_Office'], columns=df['Produtividade_bin'], margins=True)

In [ ]:
# Percentuais por regime de trabalho
(pd.crosstab(index=df['Home_Office'], columns=df['Produtividade_bin'], normalize='index') * 100).round(2)

In [ ]:
tabela_home = pd.crosstab(index=df['Home_Office'], columns=df['Produtividade_bin'], normalize='index') * 100
tabela_home = tabela_home[['Baixa', 'Alta']]
tabela_home.plot.bar(stacked=True, figsize=(7, 4), edgecolor='black')
plt.title('Produtividade por regime de home-office')
plt.xlabel('Home_Office')
plt.ylabel('Percentual dentro da categoria')
plt.xticks(rotation=0)
plt.legend(title='Produtividade_bin')
plt.show()

Sim. A associação exploratória entre `Home_Office` e `Produtividade_bin` é extremamente forte nesta amostra. Entre os funcionários que não fazem home-office, **100,00%** estão no grupo de baixa produtividade. Entre os que fazem home-office, **99,13%** estão no grupo de alta produtividade. Esse padrão indica forte associação descritiva entre o regime de home-office e a classificação de produtividade.

## 8. Questão (g): variáveis com associação aparentemente mais forte

Com base nas tabelas resumo, nos boxplots, gráficos de violino e tabelas de dupla entrada, as variáveis que parecem apresentar associação mais forte com `Produtividade_bin` são:

1. **Home_Office**: a separação é quase perfeita. Dos funcionários sem home-office, 100% estão em baixa produtividade; dos funcionários com home-office, 99,13% estão em alta produtividade.
2. **Horas_Trabalhadas**: o grupo de alta produtividade trabalha, em média, 43,79 horas, enquanto o grupo de baixa produtividade trabalha, em média, 34,00 horas. A diferença média é de 9,79 horas, com pouca sobreposição visual nos boxplots.
3. **Salário**: embora não tenha sido um dos pontos centrais da questão (f), também aparece diferença relevante: o grupo de alta produtividade tem salário médio de 9527,15 e mediana de 7125,84, enquanto o grupo de baixa produtividade tem salário médio de 6558,61 e mediana de 3485,34. Porém, como `Salário` apresenta alta dispersão e assimetria, sua interpretação exige cautela.

As variáveis `Satisfação`, `Tempo_Empresa`, `Cursos_Realizados`, `Idade` e `Gênero` parecem apresentar associação fraca ou pouco evidente com a produtividade binária. `Departamento` apresenta uma diferença mais perceptível para RH, mas ainda assim com intensidade menor do que `Home_Office` e `Horas_Trabalhadas`.

## 9. Questão (h): limitações da análise exploratória e análises complementares

Esta análise é **exploratória** e, portanto, permite identificar padrões, diferenças descritivas e possíveis associações entre variáveis. Porém, ela não permite concluir, sozinha, que uma variável causa maior ou menor produtividade. Por exemplo, embora `Home_Office` esteja fortemente associado à alta produtividade nesta amostra, não é possível afirmar apenas com estes gráficos e tabelas que o home-office causa aumento de produtividade. Pode haver outras variáveis relacionadas, como tipo de função, perfil dos funcionários, política de alocação da empresa ou critérios de seleção para home-office.

Além disso, as associações observadas não necessariamente possuem **significância estatística**, pois não foram aplicados procedimentos inferenciais. As diferenças observadas podem depender da amostra analisada, da forma como os dados foram gerados, do ponto de corte escolhido para a mediana e da presença de variáveis de confusão. A transformação de `Produtividade` em variável binária também simplifica a informação, pois funcionários logo acima e logo abaixo da mediana podem ser bastante parecidos, mas passam a ser classificados em grupos distintos.

Para uma análise confirmatória, poderiam ser consideradas análises complementares como: testes de hipóteses para comparar grupos, medidas formais de associação entre variáveis qualitativas, intervalos de confiança e modelos estatísticos que permitam avaliar simultaneamente o efeito de várias variáveis sobre a produtividade. Como `Produtividade_bin` é binária, uma análise posterior poderia estudar a probabilidade de alta produtividade em função das demais características. Se a variável `Produtividade` original fosse mantida como quantitativa, também seria possível analisar sua relação com as variáveis explicativas sem perder informação pelo corte na mediana. Essas análises confirmatórias teriam a implicação de avaliar se os padrões encontrados visualmente são consistentes do ponto de vista estatístico, controlando melhor a incerteza amostral e possíveis fatores de confusão.

## 10. Conclusão geral

A análise exploratória sugere que as características mais associadas à produtividade binária são `Home_Office` e `Horas_Trabalhadas`. Funcionários em regime de home-office aparecem quase totalmente no grupo de alta produtividade, enquanto funcionários sem home-office aparecem no grupo de baixa produtividade. Além disso, funcionários de alta produtividade trabalham mais horas, em média, do que os de baixa produtividade. Por outro lado, variáveis como satisfação, tempo de empresa, idade, gênero e número de cursos realizados não apresentam diferenças tão expressivas entre os grupos. O departamento RH apresenta maior proporção de baixa produtividade, mas essa associação é menos forte do que as observadas para home-office e horas trabalhadas.